# Budgerigar：连续神经复读基线

模型不包含 LISTEN/END/READ 状态机。输入与输出处于同一连续时间轴：听句子时学习输出静默，内容完整后保留一小段自然思考时间，再通过网络隐状态产生固定声线复读。VAD 是输入证据，不是启动规则。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位特征
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
assert FEATURE_MANIFEST.is_file(), '请先完成特征 notebook'
print(FEATURE_MANIFEST)

In [ ]:
#@title 3. 审计固定声线平行 episode
TARGET_SPEAKER='arctic_slt' #@param {type:'string'}
import json,torch
from budgerigar.echo_data import load_pairs,pair_report
pairs=load_pairs(FEATURE_MANIFEST,TARGET_SPEAKER)
summary=pair_report(pairs)
print(json.dumps(summary,ensure_ascii=False,indent=2))
assert summary['by_split']['train']>0 and summary['by_split']['validation']>0
assert TARGET_SPEAKER not in summary['source_speakers']

In [ ]:
#@title 4. 检查连续时间轴（没有 action/state 标签）
from budgerigar.echo_data import EchoEpisodeDataset,feature_stats
train_pairs=[pair for pair in pairs if pair.split=='train']
stats=feature_stats(train_pairs)
preview=EchoEpisodeDataset(train_pairs[:2],stats,thinking_frames=(16,28),preload=True)
inputs,outputs,voice,source_id=preview[0]
print(source_id,inputs.shape,outputs.shape,voice.shape)
assert inputs.shape[0]==outputs.shape[0]==voice.shape[0]
print('前段平均发声监督:',float(voice[:len(voice)//2].mean()),'全段:',float(voice.mean()))

In [ ]:
#@title 5. T4 smoke training
MAX_STEPS=300 #@param {type:'integer'}
BATCH_SIZE=6 #@param {type:'integer'}
if not torch.cuda.is_available(): raise RuntimeError('请选择 GPU runtime')
from budgerigar.train_echo import EchoTrainingConfig,train_neural_echo
RUN_DIR=WORK_ROOT/'checkpoints'/f'neural_echo_{TARGET_SPEAKER}_{FEATURE_FINGERPRINT}'
report=train_neural_echo(FEATURE_MANIFEST,RUN_DIR,EchoTrainingConfig(target_speaker=TARGET_SPEAKER,batch_size=BATCH_SIZE,max_steps=MAX_STEPS))
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 6. 曲线与产物检查
import matplotlib.pyplot as plt
history=report['history']
plt.plot([x['step'] for x in history],[x['train_loss'] for x in history],marker='o',label='train')
plt.plot([x['step'] for x in history],[x['validation_loss'] for x in history],marker='o',label='validation')
plt.legend();plt.grid();plt.xlabel('step');plt.ylabel('loss');plt.show()
for name in ('last.pt','best.pt','training_report.json'):
    path=RUN_DIR/name; assert path.is_file(),path; print(name,path.stat().st_size)

In [ ]:
#@title 7. 保存运行元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(RUN_DIR/'run_metadata.json',FEATURE_MANIFEST,{'behavior':'continuous_neural_listen_then_repeat','target_speaker':TARGET_SPEAKER,'best_validation_loss':report['best_validation_loss']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))